### 特徵抓取與模型訓練

In [6]:
#!/usr/bin/env python3
import time, json
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, top_k_accuracy_score
import warnings
import pickle


warnings.filterwarnings("ignore", category=UserWarning, module='xgboost')
warnings.filterwarnings("ignore", category=DeprecationWarning)

class FPNode:
    """A node in an FP-tree."""
    def __init__(self, code=None):
        self.code = code
        self.count = 0
        self.children = {}

def main():
    results = {}
    # --- 1. 載入資料---
    t0 = time.time()
    # transfer t0 to utc -4 time
    print("Loading dataset...", time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime(t0 - 14400)))

    df = pd.read_csv('./data/dataset_10000.csv', sep='delimiter', header=None, engine='python')
    txns = df[0].str.split(',').tolist()
    # txns = [[i for i in t if i != '-1' and i != ''] for t in txns]
    txns = [
        [item.strip() for item in txn
        if item.strip() and item.strip() != '-1']
        for txn in txns
    ]
    freq_counter = Counter(i for t in txns for i in t)
    unique_items = sorted(freq_counter)
    print(">> Unique items (before encoding):", unique_items, "count =", len(unique_items), flush=True)

    enc = LabelEncoder(); enc.fit(unique_items)
    N = len(txns)
    results['num_txns'] = N
    results['num_items'] = len(unique_items)
    results['preproc_time_s'] = time.time() - t0

    # --- 2. 特徵抽取 ---
    t1 = time.time()
    print("Extracting features...", time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime(t1 - 14400)))
    # real_root = type('FPNode', (), {'__init__': lambda s,code=None: setattr(s, 'children', {}) or setattr(s, 'count',0) or setattr(s,'code',code)})()
    real_root = FPNode()
    real_root.count = 0
    features, labels = [], []
    for txn in txns:
        seq = sorted(txn, key=lambda x:-freq_counter[x])
        codes = enc.transform(seq)
        node = real_root; node.count += 1; prefix = []
        for c in codes:
            # build feature vector
            pv = np.zeros(len(unique_items), int)
            for p in prefix: pv[p] = 1
            
            cc = np.zeros(len(unique_items), int)
            for ch, nd in node.children.items(): cc[ch] = nd.count
            
            pc, pl = node.count, len(prefix)
            
            item = enc.inverse_transform([c])[0]
            
            p_c = freq_counter[item]/N


            # p_p = freq_counter[prefix[-1]]/N if prefix else 1.0
            if prefix:
                last_code = prefix[-1]
                last_item = enc.inverse_transform([last_code])[0]
                p_p = freq_counter[last_item] / N
            else:
                p_p = 1.0
            
            edge_ct = node.children.get(c, type(node)()).count
            p_edge = edge_ct / N
            p_bound = p_c * p_p * p_edge

            feat = np.concatenate([pv, cc, [pc], [pl], [p_bound]])
            features.append(feat); labels.append(c)

            # 插入節點
            if c not in node.children:
                nd = type(node)(code=c)
                nd.count = 0
                node.children[c] = nd
            node = node.children[c]; node.count += 1
            prefix.append(c)
            
            # if len(features) < 5:
            #     print("DBG:", item, p_c, p_p, p_edge, p_bound, flush=True)

    real_root.count = N
    X = np.array(features); y = np.array(labels)
    results['feature_count'] = X.shape[0]
    results['feature_time_s'] = time.time() - t1

    

    # --- 3. XGBoost 依商品拆分模型 ---
    t2 = time.time()
    print("Training models...", time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime(t2 - 14400)))
    tr, te = train_test_split(np.arange(len(y)), test_size=0.2, random_state=42)
    X_tr, y_tr = X[tr], y[tr]
    X_te, y_te = X[te], y[te]

    item_models = {}
    for code in range(len(unique_items)):
        yb = (y_tr == code).astype(int)
        if yb.sum() < 30: continue
        mdl = XGBClassifier(
           objective='binary:logistic',
           use_label_encoder=False, eval_metric='logloss',
           n_estimators=100, max_depth=6, learning_rate=0.1
        )
        mdl.fit(X_tr, yb)
        item_models[code] = mdl
    results['num_trained_models'] = len(item_models)
    results['train_time_s'] = time.time() - t2

    # --- 4. 預測 & 評估 ---
    t3 = time.time()
    print("Evaluating models...", time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime(t3 - 14400)))
    proba = np.zeros((len(te), len(unique_items)))
    for i, idx in enumerate(te):
        xi = X[idx].reshape(1, -1)
        for c, m in item_models.items():
            proba[i, c] = m.predict_proba(xi)[0,1]
    acc1 = accuracy_score(y_te, proba.argmax(axis=1))
    # acc3 = top_k_accuracy_score(y_te, proba, k=3)
    results.update({
        'top1_acc': acc1,
        'top2_acc': top_k_accuracy_score(y_te, proba, k=2),
        'top3_acc': top_k_accuracy_score(y_te, proba, k=3),
        'top4_acc': top_k_accuracy_score(y_te, proba, k=4),
        'top5_acc': top_k_accuracy_score(y_te, proba, k=5),
    })
    results['eval_time_s'] = time.time() - t3

    # --- 5. 輸出結果 ---
    # 存 JSON
    with open('results_summary.json', 'w') as f:
        json.dump(results, f, indent=2)
    # Bash 輸出
    print("--- Results Summary ---")
    for k,v in results.items():
        print(f"{k:20s}: {v}")

    # --- 6. 保存模型與所需物件 ---
    
    print("Saving model artifacts...", flush=True)
    artifacts = {
        'models': item_models,
        'encoder': enc,
        'freq_counter': freq_counter,
        'real_root': real_root,
        'num_total_txns': N,
        'unique_items': unique_items
    }
    
    with open('fp_model_artifacts.pkl', 'wb') as f:
        pickle.dump(artifacts, f)
        
    print("Artifacts saved to fp_model_artifacts.pkl")

if __name__ == '__main__':
    main()


Loading dataset... 2025-07-02 15:58:18
>> Unique items (before encoding): ['Bread', 'Butter', 'Cheese', 'Coffee Powder', 'Ghee', 'Lassi', 'Milk', 'Panner', 'Sugar', 'Sweet', 'Tea Powder', 'Yougurt'] count = 12
Extracting features... 2025-07-02 15:58:18
Training models... 2025-07-02 15:58:33
Evaluating models... 2025-07-02 15:58:39
--- Results Summary ---
num_txns            : 12526
num_items           : 12
preproc_time_s      : 0.21648573875427246
feature_count       : 65713
feature_time_s      : 15.12514853477478
num_trained_models  : 12
train_time_s        : 5.762255907058716
top1_acc            : 0.9669786197976109
top2_acc            : 0.9918587841436506
top3_acc            : 0.9983261051510309
top4_acc            : 0.9991630525755155
top5_acc            : 0.9997717416115042
eval_time_s         : 111.28232502937317
Saving model artifacts...
Artifacts saved to fp_model_artifacts.pkl


### 產生候選集

In [159]:
#!/usr/bin/env python3
import pickle
import numpy as np
import heapq

class FPNode:
    """A node in an FP-tree."""
    def __init__(self, code=None):
        self.code = code
        self.count = 0
        self.children = {}
        
def generate_features_for_prefix(prefix_codes, artifacts):
    """
    為給定的前綴生成預測下一步驟所需的特徵。
    這個函數一次性準備好上下文，而不是為每個可能的子節點重複計算。
    """
    enc = artifacts['encoder']
    real_root = artifacts['real_root']
    freq_counter = artifacts['freq_counter']
    N = artifacts['num_total_txns']
    num_unique_items = len(artifacts['unique_items'])

    node = real_root
    # 根據 prefix_codes 遍歷樹，找到當前節點
    for code in prefix_codes:
        if code in node.children:
            node = node.children[code]
        else:
            # 如果路徑在訓練時的樹中不存在，則無法繼續生成
            return None, None 

    # --- 準備特徵 ---
    # 1. 前綴向量 (pv)
    pv = np.zeros(num_unique_items, int)
    for p in prefix_codes:
        pv[p] = 1

    # 2. 子節點計數向量 (cc)
    cc = np.zeros(num_unique_items, int)
    for ch, nd in node.children.items():
        cc[ch] = nd.count

    # 3. 父節點計數 (pc) 和 前綴長度 (pl)
    pc = node.count
    pl = len(prefix_codes)
    
    # 4. 前綴最後一個項目的機率 P(p)
    if prefix_codes:
        last_code = prefix_codes[-1]
        last_item = enc.inverse_transform([last_code])[0]
        p_p = freq_counter[last_item] / N
    else:
        # 如果是空前綴，父機率設為1.0
        p_p = 1.0

    base_features = (pv, cc, pc, pl, p_p)
    return base_features, node


def generate_candidate_patterns(beam_width=5, max_length=5, start_with_prefix=None):
    """
    使用Beam Search從訓練好的模型生成候選樣式
    bug fix: start_with_prefix 的排序問題
    """
    # print("Loading model artifacts...")
    with open('fp_model_artifacts.pkl', 'rb') as f:
        artifacts = pickle.load(f)

    item_models = artifacts['models']
    enc = artifacts['encoder']
    freq_counter = artifacts['freq_counter']
    N = artifacts['num_total_txns']
    num_unique_items = len(artifacts['unique_items'])

    candidate_patterns = []

    # 使用一個集合來追蹤已加入的樣式的「標準型」(排序後的tuple)
    # 這能確保最終結果中，每個 itemset 都是獨一無二的 -- bug fix
    canonical_candidate_set = set()

    if start_with_prefix and len(start_with_prefix) > 0:
        print(f"Starting generation with a specific prefix: {start_with_prefix}")
        try:
            prefix_codes = enc.transform(start_with_prefix).tolist()
            
            # 根據商品頻率對前綴進行排序，以匹配FP-Tree中的路徑順序 # bug fix
            prefix_codes.sort(key=lambda code: freq_counter[enc.inverse_transform([code])[0]], reverse=True)

            # 確保prefix的順序是正確的，並且不會重複
            canonical_candidate_set.add(tuple(sorted(prefix_codes)))

            if max_length <= len(prefix_codes):
                print("Warning: max_length is not greater than the length of start_with_prefix.")
                return [enc.inverse_transform(prefix_codes).tolist()]

            beam = [(0.0, prefix_codes)]
            candidate_patterns.append(prefix_codes)
            num_generations = max_length - len(prefix_codes)

        except ValueError as e:
            print(f"Error: An item in the prefix is not in the model's vocabulary. Details: {e}")
            return []
    else:
        print("Starting generation from scratch...")
        beam = [(0.0, [])]
        num_generations = max_length
    
    print(f"Starting generation with beam_width={beam_width}, generating up to {num_generations} new items")
    
    for length in range(num_generations):
        # print(f"Generating new items (step {length + 1}/{num_generations})...")
        next_beam = []
        
        # 擴展和剪枝的邏輯
        for score, path_codes in beam:
            base_features, node = generate_features_for_prefix(path_codes, artifacts)

            if base_features is None:
                # 這條路徑走到了盡頭（在訓練樹中不存在），將其加入最終結果
                # bug fix: 避免重複的路徑，就直接跳過，不進行任何添加操作
                # if path_codes not in candidate_patterns:
                #     candidate_patterns.append(path_codes)
                continue

            pv, cc, pc, pl, p_p = base_features

            for c in range(num_unique_items):
                if c in path_codes or c not in item_models:
                    continue

                # --- 為這個特定的(路徑, c)組合完成特徵向量 ---
                item = enc.inverse_transform([c])[0]
                p_c = freq_counter[item] / N if N > 0 else 0
                
                edge_ct = node.children.get(c, type('FPNode', (), {'count': 0})()).count
                p_edge = edge_ct / N if N > 0 else 0
                p_bound = p_c * p_p * p_edge
                
                # 組合最終特徵向量
                feat = np.concatenate([pv, cc, [pc], [pl], [p_bound]]).reshape(1, -1)
                
                # 套模型預測機率
                model = item_models[c]
                proba = model.predict_proba(feat)[0, 1]
                
                if proba > 1e-5: # 避免 log(0)
                # if proba > 1e-6:
                    new_path = path_codes + [c]
                    # 分數是負對數概似總和
                    new_score = score - np.log(proba)
                    heapq.heappush(next_beam, (new_score, new_path))
        
        # 從所有可能的下一步中，選出分數最好的 K 個，成為新的 beam
        # 並將上一輪的路徑加入candidate_patterns
        beam = heapq.nsmallest(beam_width, next_beam)

        # 如果無法再擴展，提前終止
        if not beam: 
            print("No more paths to expand.")
            break
            
        # for s, p in beam:
        #     if p not in candidate_patterns:
        #         candidate_patterns.append(p)

        # bug fix: 避免重複的路徑
        # 遍歷當前 beam 中的最佳路徑，並以 order-agnostic 的方式加入最終列表
        for s, path_codes in beam:
            # 創建路徑的標準型 (sorted tuple)
            canonical_path = tuple(sorted(path_codes))
            
            # 檢查標準型是否已經存在
            if canonical_path not in canonical_candidate_set:
                # 如果不存在，則將標準型和原始路徑都加入
                canonical_candidate_set.add(canonical_path)
                candidate_patterns.append(path_codes) # 加入原始順序的路徑

    # print("\nGeneration complete. Deduplicating final results...")
    
    # final_unique_patterns = []
    # canonical_set = set()
    
    # for path_codes in canonical_candidate_set:
    #     canonical_path = tuple(sorted(path_codes))
    #     if canonical_path not in canonical_set:
    #         canonical_set.add(canonical_path)
    #         final_unique_patterns.append(path_codes) # 只添加第一次遇到的原始順序

    # 將編碼轉回商品名稱
    decoded_patterns = []
    for p_codes in candidate_patterns:
    # for p_codes in final_unique_patterns:
        if p_codes:
            decoded_patterns.append(enc.inverse_transform(p_codes).tolist())
        
    return decoded_patterns

if __name__ == '__main__':
    NUM_CANDIDATE = 95  # beam_width
    PATTERN_LENGTH = 3  # max_length
    # 生成候選樣式
    candidates = generate_candidate_patterns(beam_width=NUM_CANDIDATE, max_length=PATTERN_LENGTH)
    print(f"\n--- Generated Candidate Patterns (beam_width={NUM_CANDIDATE}, max_length={PATTERN_LENGTH})---")
    # 按照長度排序輸出
    candidates.sort(key=len, reverse=True)
    # for pattern in candidates:
    #     print(pattern)
    print("Total patterns generated:", len(candidates))

c:\Users\zoezo\anaconda3\lib\site-packages\ipykernel\ipkernel.py:287: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


Starting generation from scratch...
Starting generation with beam_width=95, generating up to 3 new items

--- Generated Candidate Patterns (beam_width=95, max_length=3)---
Total patterns generated: 173


### 驗證 min_support

In [164]:
def validate_patterns(candidate_patterns, min_support_count=100):
    """
    驗證候選樣式的真實支持度。
    """
    print("\n--- Validating Candidate Patterns ---")
    
    # 重新載入原始交易數據的訓練集

    import pandas as pd
    df = pd.read_csv('./data/dataset_10000.csv', sep='delimiter', header=None, engine='python')
    txns = df[0].str.split(',').tolist()
    txns = [
        {item.strip() for item in txn if item.strip() and item.strip() != '-1'}
        for txn in txns
    ]
    # 只用訓練集 0.8 的交易數據
    txns_train = txns[:int(len(txns) * 0.8)]

    frequent_patterns = {}
    support_not_select = []
    for pattern in candidate_patterns:
        support_count = 0
        pattern_set = set(pattern)
        for txn_set in txns_train:
            if pattern_set.issubset(txn_set):
                support_count += 1
        support = support_count/len(txns_train)
        if support_count >= min_support_count*len(txns_train):
        # if support_count >= min_support_count:
            frequent_patterns[tuple(pattern)] = support # (support_count / len(txns_train))
        else:
            # print(f"{pattern}: {min_support_count} > Support count: {(support):.4f}")
            support_not_select.append(support)

    # the average support of the patterns not selected
    if support_not_select:
        avg_support_not_select = sum(support_not_select) / len(support_not_select)
        print(f"Average support of patterns not selected: {avg_support_not_select:.4f}")
        # the highest support of the patterns not selected
        max_support_not_select = max(support_not_select)
        print(f"Highest support of patterns not selected: {max_support_not_select:.4f}")
        # the lowest support of the patterns not selected
        min_support_not_select = min(support_not_select)
        print(f"Lowest support of patterns not selected: {min_support_not_select:.4f}")
        # median support of the patterns not selected
        median_support_not_select = np.median(support_not_select)
        print(f"Median support of patterns not selected: {median_support_not_select:.4f}")

        print(f"Total support not selected: {len(support_not_select)} patterns")


    print(f"Found {len(frequent_patterns)} frequent patterns with support >= {min_support_count} in {len(txns_train)} transactions.")
    # 通過驗證的比例
    print(f"Validation ratio: {len(frequent_patterns) / len(candidate_patterns):.2%}: {len(frequent_patterns)}/{len(candidate_patterns)}")
    # print(len(candidate_patterns), "candidate patterns generated.")
    # print(len(frequent_patterns), "frequent patterns found after validation.")

    # 各長度通握驗證的比例
    lengths = [len(p) for p in candidate_patterns]
    length_counter = Counter(lengths)
    # print("\n--- Length-wise Validation Ratios ---")
    for length, count in length_counter.items():
        valid_count = sum(1 for p in frequent_patterns if len(p) == length)
        ratio = valid_count / count if count > 0 else 0
        print(f"Length {length}: {valid_count}/{count} = {ratio:.2%}")
    

    return frequent_patterns

if __name__ == '__main__':
    # ... (前面的 generate_candidate_patterns 呼叫) ...
    
    # 設定你想要的最小支持度計數，例如 1% 的交易量
    # MIN_SUPPORT_COUNT = 100 
    MIN_SUPPORT = 0.1 # 0.05 
    
    final_fps = validate_patterns(candidates, min_support_count=MIN_SUPPORT)
    # final_fps = validate_patterns(candidates, min_support_count=MIN_SUPPORT_COUNT)
    
    # print(f"\n--- Final Frequent Patterns (Support >= {MIN_SUPPORT_COUNT}) ---")
    # print(f"\n--- Final Frequent Patterns (min_support = {MIN_SUPPORT}) ---")
    # 根據支持度從高到低排序
    sorted_fps = sorted(final_fps.items(), key=lambda item: item[1], reverse=True)
    
    for pattern, support in sorted_fps:
        if len(pattern) < 2:
            continue
        else:
            # print(f"Pattern: {list(pattern)}, Support: {(support):.4f}")
            continue

c:\Users\zoezo\anaconda3\lib\site-packages\ipykernel\ipkernel.py:287: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)



--- Validating Candidate Patterns ---
Average support of patterns not selected: 0.0941
Highest support of patterns not selected: 0.0994
Lowest support of patterns not selected: 0.0884
Median support of patterns not selected: 0.0941
Total support not selected: 94 patterns
Found 79 frequent patterns with support >= 0.1 in 10020 transactions.
Validation ratio: 45.66%: 79/173
Length 3: 1/95 = 1.05%
Length 2: 66/66 = 100.00%
Length 1: 12/12 = 100.00%


### FP growth 

In [128]:
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import fpgrowth

def run_fp_growth(min_support_threshold=0.01):
    """
    執行 FP-Growth 演算法並返回結果。
    """
    print("\n--- Running Traditional FP-Growth ---")
    
    # 1. 載入並整理原始交易數據
    df_raw = pd.read_csv('./data/dataset_10000.csv', sep='delimiter', header=None, engine='python')
    txns = df_raw[0].str.split(',').tolist()
    txns = [
        [item.strip() for item in txn if item.strip() and item.strip() != '-1']
        for txn in txns
    ]
    # 只用訓練集 0.8 的交易數據
    txns = txns[:int(len(txns) * 0.8)]

    # 2. 將數據轉換為 one-hot 編碼格式
    te = TransactionEncoder()
    te_ary = te.fit(txns).transform(txns)
    df_onehot = pd.DataFrame(te_ary, columns=te.columns_)

    # 3. 執行 FP-Growth
    print(f"Finding frequent patterns with min_support >= {min_support_threshold}...")
    frequent_itemsets_fp = fpgrowth(df_onehot, min_support=min_support_threshold, use_colnames=True)
    
    print(f"FP-Growth found {len(frequent_itemsets_fp)} frequent patterns.")
    
    return frequent_itemsets_fp

# --- 主程式中呼叫 ---
if __name__ == '__main__':
    # ... (您原有的 ML 生成和驗證程式碼) ...
    
    # 現在執行 FP-Growth
    fp_results = run_fp_growth(min_support_threshold=MIN_SUPPORT)
    print("\n--- FP-Growth Top 5 Results ---")
    # print(fp_results.sort_values(by="support", ascending=False).head())
    # itemset >= 2
    print(fp_results[fp_results['itemsets'].apply(lambda x: len(x) >= 2)].sort_values(by="support", ascending=False).head())

c:\Users\zoezo\anaconda3\lib\site-packages\ipykernel\ipkernel.py:287: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)



--- Running Traditional FP-Growth ---
Finding frequent patterns with min_support >= 0.1...
FP-Growth found 82 frequent patterns.

--- FP-Growth Top 5 Results ---
     support               itemsets
72  0.207285         (Sweet, Bread)
43  0.206587         (Lassi, Sweet)
73  0.206387        (Sugar, Butter)
21  0.204790  (Coffee Powder, Ghee)
19  0.204491        (Sweet, Butter)


### compare

In [129]:
# fp_results 是您從 run_fp_growth 得到的 DataFrame
print("min_support_threshold:", MIN_SUPPORT)
# 計算每個樣式的長度
fp_results['length'] = fp_results['itemsets'].apply(lambda x: len(x))

# 按長度統計數量
fp_length_counts = fp_results['length'].value_counts().sort_index()

print("\n--- FP-Growth Pattern Counts by Length ---")
print(fp_length_counts)


# 將這個結果與您 ML 方法找到的96個樣式的長度分佈做比較

ml_patterns_df = pd.DataFrame({'itemsets': final_fps.keys()})
ml_patterns_df['length'] = ml_patterns_df['itemsets'].apply(len)
ml_length_counts = ml_patterns_df['length'].value_counts().sort_index()
print("\n--- ML Method Pattern Counts by Length ---")
print(ml_length_counts)

# TP, TN, FP, FN 計算
def calculate_tp_tn_fp_fn(ml_patterns, fp_patterns):
    """
    計算 TP, TN, FP, FN。
    """
    ml_set = set(tuple(sorted(p)) for p in ml_patterns)
    fp_set = set(tuple(sorted(p)) for p in fp_patterns)

    tp = len(ml_set & fp_set)  # 真陽性
    fp = len(ml_set - fp_set)  # 假陽性
    fn = len(fp_set - ml_set)  # 假陰性
    tn = 0  # 傳統 FP-Growth 沒有負樣本，所以 TN 為 0

    return tp, tn, fp, fn
def print_tp_tn_fp_fn(tp, tn, fp, fn):
    """
    輸出 TP, TN, FP, FN 的結果。
    """
    # print("\n--- TP, TN, FP, FN Results ---")
    print(f"True Positives (TP): {tp}")
    # print(f"True Negatives (TN): {tn}")
    print(f"False Positives (FP): {fp}")
    print(f"False Negatives (FN): {fn}")

tp, tn, fp, fn = calculate_tp_tn_fp_fn(ml_patterns=final_fps.keys(), fp_patterns=fp_results['itemsets'])
print_tp_tn_fp_fn(tp, tn, fp, fn)

# precision, recall, f1_score
def calculate_print_precision_recall_f1(tp, fp, fn):
    """
    計算精確率、召回率和 F1 分數。
    """
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    # print("\n--- Precision, Recall, F1 Score ---")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1 Score: {f1_score:.4f}")
    return precision, recall, f1_score

p, r,f1 = calculate_print_precision_recall_f1(tp, fp, fn)

min_support_threshold: 0.1

--- FP-Growth Pattern Counts by Length ---
length
1    12
2    66
3     4
Name: count, dtype: int64

--- ML Method Pattern Counts by Length ---
length
1    12
2    66
3     4
Name: count, dtype: int64
True Positives (TP): 82
False Positives (FP): 0
False Negatives (FN): 0
Precision: 1.0000
Recall: 1.0000
F1 Score: 1.0000


c:\Users\zoezo\anaconda3\lib\site-packages\ipykernel\ipkernel.py:287: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


### compare - 給定prefix

In [115]:
# 定義想查詢的prefix
# 使用 frozenset 來匹配 itemsets
prefix_to_find = frozenset(['Milk', 'Ghee'])

# 篩選
mask = fp_results['itemsets'].apply(lambda x: x.issuperset(prefix_to_find))
filtered_fp_results = fp_results[mask]

print(f"\n--- FP-Growth patterns containing {list(prefix_to_find)} ---")
# print(filtered_fp_results.sort_values(by="support", ascending=False))

print(len(filtered_fp_results), "patterns found with prefix", list(prefix_to_find))

# count by length
# print("\n--- Count by Length of Filtered Patterns ---")
filtered_fp_results['length'] = filtered_fp_results['itemsets'].apply(lambda x: len(x))
filtered_length_counts = filtered_fp_results['length'].value_counts().sort_index()
print(filtered_length_counts)


# ML 方法生成候選樣式的程式碼
print(f"\n--- ML Method Patterns with Prefix {list(prefix_to_find)} ---")
ml_prefix_results = generate_candidate_patterns(
    beam_width=150, #120 #50, #20,
    max_length=6,
    # start_with_prefix=['Milk', 'Ghee']
    # start_with_prefix=['Ghee', 'Milk']
    start_with_prefix=list(prefix_to_find)
)
# print(ml_prefix_results)
# ml_prefix_results 進行驗證
MIN_SUPPORT = 0.01
ml_prefix_results = validate_patterns(ml_prefix_results, min_support_count=MIN_SUPPORT)
# print(len(ml_prefix_results))


c:\Users\zoezo\anaconda3\lib\site-packages\ipykernel\ipkernel.py:287: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)
<ipython-input-115-8aa182dda504>:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_fp_results['length'] = filtered_fp_results['itemsets'].apply(lambda x: len(x))



--- FP-Growth patterns containing ['Ghee', 'Milk'] ---
220 patterns found with prefix ['Ghee', 'Milk']
length
2      1
3     10
4     45
5    120
6     44
Name: count, dtype: int64

--- ML Method Patterns with Prefix ['Ghee', 'Milk'] ---
Starting generation with a specific prefix: ['Ghee', 'Milk']
Starting generation with beam_width=150, generating up to 4 new items

--- Validating Candidate Patterns ---
Found 206 frequent patterns with support >= 0.01 in 10020 transactions.
Validation ratio: 63.19%: 206/326
Length 2: 1/1 = 100.00%
Length 3: 10/10 = 100.00%
Length 4: 45/45 = 100.00%
Length 5: 120/120 = 100.00%
Length 6: 30/150 = 20.00%


In [116]:
# ML結果與 FP-Growth 的篩選結果進行比較
print("\n--- Comparing ML Method Patterns with FP-Growth Filtered Patterns ---")
# 計算 ML 方法與 FP-Growth 篩選結果的交集
# print(len(ml_prefix_results))
ml_set = set(tuple(sorted(p)) for p in ml_prefix_results)
fp_set = set(tuple(sorted(p)) for p in filtered_fp_results['itemsets'])

intersection = ml_set.intersection(fp_set)
print(f"common patterns: {len(intersection)}")
#TP, FP, FN, TN
# print(len(ml_set), "ML patterns")
TP = len(intersection)
FP = len(ml_set) - TP
FN = len(fp_set) - TP
print(f"TP: {TP}, FP: {FP}, FN: {FN}")

# precision, recall, f1_score
precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
recall = TP / (TP + FN) if (TP + FN) > 0 else 0.0
f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
print(f"Precision: {precision:.4f}, \nRecall: {recall:.4f}, \nF1 Score: {f1_score:.4f}")

# 按長度統計交集
intersection_lengths = [len(p) for p in intersection]
intersection_length_counts = Counter(intersection_lengths)
print("\n--- Intersection Length Counts ---")   
# sort by length
intersection_length_counts = dict(sorted(intersection_length_counts.items()))
for length, count in intersection_length_counts.items():
    print(f"Length {length}: {count} patterns")



--- Comparing ML Method Patterns with FP-Growth Filtered Patterns ---
common patterns: 206
TP: 206, FP: 0, FN: 14
Precision: 1.0000, 
Recall: 0.9364, 
F1 Score: 0.9671

--- Intersection Length Counts ---
Length 2: 1 patterns
Length 3: 10 patterns
Length 4: 45 patterns
Length 5: 120 patterns
Length 6: 30 patterns


c:\Users\zoezo\anaconda3\lib\site-packages\ipykernel\ipkernel.py:287: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


### 研究輸出

In [110]:
from collections import Counter

# Assuming ml_prefix_results is your dictionary of {pattern_tuple: support}
# with 115 items.

# Create a list of sorted tuples for every pattern
sorted_patterns = [tuple(sorted(p)) for p in ml_prefix_results.keys()]

# Count the occurrences of each sorted pattern
pattern_counts = Counter(sorted_patterns)

# Find and print the patterns that appeared more than once
print("--- Finding Duplicate Itemsets with Different Orderings ---")
duplicates_found = False
for pattern, count in pattern_counts.items():
    if count > 1:
        duplicates_found = True
        print(f"Itemset {pattern} appeared {count} times with different orderings.")

if not duplicates_found:
    print("No differently-ordered duplicates were found.")

# Itemset ('Butter', 'Ghee', 'Milk', 'Tea Powder') appeared 2 times with different orderings.
# Itemset ('Cheese', 'Ghee', 'Milk', 'Tea Powder') appeared 2 times with different orderings.
# Itemset ('Ghee', 'Milk', 'Panner', 'Tea Powder') appeared 2 times with different orderings.
# Itemset ('Ghee', 'Milk', 'Sugar', 'Tea Powder') appeared 2 times with different orderings.
# Itemset ('Ghee', 'Lassi', 'Milk', 'Tea Powder') appeared 2 times with different orderings.

--- Finding Duplicate Itemsets with Different Orderings ---
No differently-ordered duplicates were found.


c:\Users\zoezo\anaconda3\lib\site-packages\ipykernel\ipkernel.py:287: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


In [68]:
from collections import defaultdict

def find_original_duplicate_orders(ml_results_dict):
    """
    找出並展示造成重複的原始、未排序的樣式。
    """
    # ml_results_dict is the dictionary with 115 items, e.g., {('Ghee', 'Milk', ...): support}
    
    # 1. 找出哪些排序後的樣式是重複的
    sorted_patterns = [tuple(sorted(p)) for p in ml_results_dict.keys()]
    pattern_counts = Counter(sorted_patterns)
    duplicate_sorted_patterns = {p for p, count in pattern_counts.items() if count > 1}

    if not duplicate_sorted_patterns:
        print("No duplicates to investigate.")
        return

    # 2. 建立一個字典來收集原始順序
    # key 是排序後的樣式, value 是一個原始順序的列表
    original_orders_map = defaultdict(list)

    # 3. 遍歷原始的115個結果
    for original_pattern in ml_results_dict.keys():
        sorted_pattern = tuple(sorted(original_pattern))
        # 如果這個樣式是我們已知的重複項之一
        if sorted_pattern in duplicate_sorted_patterns:
            original_orders_map[sorted_pattern].append(original_pattern)

    # 4. 打印結果
    print("\n--- Original Orders of Duplicate Itemsets ---")
    for sorted_pattern, original_list in original_orders_map.items():
        print(f"\nItemset: {sorted_pattern}")
        print("  Found these different orderings:")
        for original in original_list:
            print(f"  - {original}")

# --- 在您的主程式區塊中呼叫 ---
if __name__ == '__main__':
    # ... 假設 ml_prefix_results 已經被定義為包含115個樣式的字典 ...
    # ml_prefix_results = validate_patterns(...) 
    
    find_original_duplicate_orders(ml_prefix_results)


--- Original Orders of Duplicate Itemsets ---

Itemset: ('Butter', 'Ghee', 'Milk', 'Tea Powder')
  Found these different orderings:
  - ('Milk', 'Ghee', 'Butter', 'Tea Powder')
  - ('Milk', 'Ghee', 'Tea Powder', 'Butter')

Itemset: ('Cheese', 'Ghee', 'Milk', 'Tea Powder')
  Found these different orderings:
  - ('Milk', 'Ghee', 'Cheese', 'Tea Powder')
  - ('Milk', 'Ghee', 'Tea Powder', 'Cheese')

Itemset: ('Ghee', 'Milk', 'Panner', 'Tea Powder')
  Found these different orderings:
  - ('Milk', 'Ghee', 'Panner', 'Tea Powder')
  - ('Milk', 'Ghee', 'Tea Powder', 'Panner')

Itemset: ('Ghee', 'Milk', 'Sugar', 'Tea Powder')
  Found these different orderings:
  - ('Milk', 'Ghee', 'Sugar', 'Tea Powder')
  - ('Milk', 'Ghee', 'Tea Powder', 'Sugar')

Itemset: ('Ghee', 'Lassi', 'Milk', 'Tea Powder')
  Found these different orderings:
  - ('Milk', 'Ghee', 'Lassi', 'Tea Powder')
  - ('Milk', 'Ghee', 'Tea Powder', 'Lassi')


c:\Users\zoezo\anaconda3\lib\site-packages\ipykernel\ipkernel.py:287: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


In [ ]:
def investigate_discrepancies(ml_results_set, fp_growth_results_set, min_support_threshold=0.01):
    """
    找出並詳細驗證 ML 方法和 FP-Growth 結果之間的差異。
    """
    print("\n" + "="*50)
    print("--- Investigating Discrepancies ---")

    # 1. 找出 False Positive (FP) 樣式：存在於 ML 結果但不存在於 FP-Growth 結果中
    false_positives = ml_results_set.difference(fp_growth_results_set)
    print(f"\nFound {len(false_positives)} False Positive patterns. Investigating each:")

    if not false_positives:
        print("No False Positives found.")
        return

    # 2. 載入原始交易數據進行精確計數
    df_raw = pd.read_csv('./data/dataset_10000.csv', sep='delimiter', header=None, engine='python')
    txns = df_raw[0].str.split(',').tolist()
    txns = [
        {item.strip() for item in txn if item.strip() and item.strip() != '-1'}
        for txn in txns
    ]
    total_txns = len(txns)
    # 根據 min_support 計算支持度計數的門檻
    support_count_threshold = total_txns * min_support_threshold

    print(f"Total Transactions: {total_txns}")
    print(f"Support Threshold: {min_support_threshold:.2f} (requires at least {support_count_threshold:.2f} transactions)")
    print("-" * 20)

    # 3. 遍歷每一個 FP 樣式，計算其確切支持度
    for i, pattern_tuple in enumerate(false_positives):
        pattern_set = set(pattern_tuple)
        exact_count = 0
        for txn_set in txns:
            if pattern_set.issubset(txn_set):
                exact_count += 1
        
        actual_support = exact_count / total_txns

        print(f"FP #{i+1}: {list(pattern_tuple)}")
        print(f"  - Exact Support Count: {exact_count}")
        print(f"  - Actual Support Ratio: {actual_support:.6f}")
        
        # 比較實際支持度和門檻
        if actual_support >= min_support_threshold:
            print("  - Status: This pattern IS frequent by exact count. The discrepancy might be in mlxtend's calculation.")
        else:
            print("  - Status: This pattern IS NOT frequent by exact count. The discrepancy might be in your validate_patterns() function.")

# --- 在您的主程式區塊中呼叫 ---
if __name__ == '__main__':
    # ... (您所有現有的比較程式碼) ...
    
    # 假設 ml_set 和 fp_set 已經被定義
    
    ml_set = set(tuple(sorted(p)) for p in ml_prefix_results)
    # fp_set = set(tuple(sorted(p)) for p in filtered_fp_results['itemsets'])
    
    investigate_discrepancies(ml_set, fp_set, min_support_threshold=0.01)


--- Investigating Discrepancies ---

Found 0 False Positive patterns. Investigating each:
No False Positives found.


c:\Users\zoezo\anaconda3\lib\site-packages\ipykernel\ipkernel.py:287: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)
